# Microsoft Agent Framework — Azure OpenAI (Responses API)

У цьому прикладі коду ви використовуватимете **Microsoft Agent Framework (MAF)** для створення простого агента на базі **Azure OpenAI** з використанням **Responses API**.

> **Примітка щодо міграції:** Цей приклад раніше використовував Semantic Kernel з GitHub Models. Він був перенесений на Microsoft Agent Framework, а GitHub Models (застарілі, припиняють роботу в липні 2026 року) замінені на Azure OpenAI, який підтримує Responses API. `OpenAIChatClient` у MAF орієнтований на стабільну кінцеву точку Azure OpenAI `/openai/v1/` і за замовчуванням використовує Responses API.

Метою цього прикладу є демонстрація кроків, які пізніше будуть застосовані в додаткових прикладах коду при реалізації різних агентських патернів.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Імпортуйте необхідні пакети Python


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Визначення інструменту

У Microsoft Agent Framework **інструмент** — це звичайна функція на Python з декоратором `@tool`, яку агент може викликати. Нижче ми визначаємо інструмент, який повертає випадкове місце відпочинку та уникає повторення попереднього.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Створення агента

Тут ми створюємо агента на ім’я `TravelAgent`.

У цьому прикладі ми використовуємо дуже прості інструкції. Не соромтеся змінювати ці інструкції, щоб спостерігати, як змінюється поведінка агента.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Запуск агента

Тепер ми можемо запустити агента. Ми створюємо `AgentSession`, щоб агент пам'ятала розмову між кроками, а потім відправляємо два `user_inputs`. Перший запитує про подорож; другий каже, що користувачеві не сподобалась пропозиція і просить іншу — агент використовує історію сесії та інструмент `get_random_destination`, щоб відповісти.

Ви можете змінювати ці повідомлення, щоб спостерігати, як агент реагує по-різному. Відповіді **передаються** токен за токеном.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
